[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AI-Hypercomputer/maxtext/blob/main/src/maxtext/examples/sft_llama3_demo_tpu.ipynb)

# Llama3.1-8B-Instruct Supervised Fine-Tuning (SFT) Demo


## Overview

This notebook demonstrates how to perform Supervised Fine-Tuning (SFT) on Llama3.1-8B-Instruct using the Hugging Face ultrachat_200k dataset with MaxText and Tunix integration for efficient training.

This notebook can run on **TPU v6e-8** or **v5p-8**.

## Prerequisites

Before running this notebook, make sure your environment is set up for the method you are using. Follow the [Run MaxText Python Notebooks on TPUs](https://maxtext.readthedocs.io/en/latest/guides/run_python_notebook.html) guide and complete all steps for your chosen method (Google Colab, VS Code, or Local Jupyter Lab) before proceeding.

If you run into issues, refer to the [Common Pitfalls & Debugging](https://maxtext.readthedocs.io/en/latest/guides/run_python_notebook.html#common-pitfalls-debugging) section of the guide.

In [ ]:
try:
  import google.colab
  print("Running the notebook on Google Colab")
  IN_COLAB = True
except ImportError:
    print("Running the notebook on Visual Studio or JupyterLab")
    IN_COLAB = False

## Installation: MaxText and Post training Dependencies

**Running the notebook on Visual Studio or JupyterLab:** Before proceeding, create a virtual environment and install the required post-training dependencies by following `Option 3: Installing [tpu-post-train]` in the [MaxText installation guide](https://maxtext.readthedocs.io/en/latest/install_maxtext.html#from-source). Once the environment is set up, ensure the notebook is running within it.

In [ ]:
if IN_COLAB:
    # Clone the MaxText repository
    !git clone https://github.com/AI-Hypercomputer/maxtext.git
    %cd maxtext

    # Install uv, a fast Python package installer
    !pip install uv
    
    # Install MaxText and post-training dependencies
    !uv pip install -e .[tpu-post-train] --resolution=lowest
    !install_tpu_post_train_extra_deps

**Session restart Instructions for Colab:**
1.  Navigate to the menu at the top of the screen.
2.  Click on **Runtime**.
3.  Select **Restart session** from the dropdown menu.

You will be asked to confirm the action in a pop-up dialog. Click on **Yes**.

## Environment Setup

In [ ]:
import datetime
import jax
import os
import subprocess
import sys
from maxtext.configs import pyconfig
from maxtext.utils.globals import MAXTEXT_PKG_DIR
from maxtext.trainers.post_train.sft import train_sft
from etils import epath


print(f"MaxText installation path: {MAXTEXT_PKG_DIR}")

In [ ]:
if not jax.distributed.is_initialized():
    jax.distributed.initialize()
print(f"JAX version: {jax.__version__}")
print(f"JAX devices: {jax.devices()}")

In [ ]:
if IN_COLAB:
    from huggingface_hub import notebook_login
    notebook_login()
else:
    from huggingface_hub import login
    login()

## Model Configurations

In [ ]:
MODEL_NAME = "llama3.1-8b-Instruct"

BASE_OUTPUT_DIRECTORY = f"{MAXTEXT_PKG_DIR}/sft_llama3_output"

# set the path to the model checkpoint (including `/0/items`) or leave empty to download from HuggingFace
MODEL_CHECKPOINT_PATH = ""
if not MODEL_CHECKPOINT_PATH:
   MODEL_CHECKPOINT_PATH = f"{BASE_OUTPUT_DIRECTORY}/llama3_checkpoint"
   print("Model checkpoint will be downloaded from HuggingFace at: ",  MODEL_CHECKPOINT_PATH)
   print("Set MODEL_CHECKPOINT_PATH if you do not wish to download the checkpoint.")

RUN_NAME = datetime.datetime.now().strftime("%Y-%m-%d-%H-%M-%S")

## Download Llama3.1-8B Model Checkpoint from Hugging Face

In [ ]:
if not epath.Path(MODEL_CHECKPOINT_PATH).exists():
    # Install torch for the conversion script
    print("Installing torch...")
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install",
            "torch", "--index-url", "https://download.pytorch.org/whl/cpu"
        ],
        check=True
    )

    # Run checkpoint conversion with environment variables
    print("Converting checkpoint from HuggingFace...")
    env = os.environ.copy()
    env["JAX_PLATFORMS"] = "cpu"

    subprocess.run(
        [
            sys.executable,
            "-m", "maxtext.checkpoint_conversion.to_maxtext",
            f"{MAXTEXT_PKG_DIR}/configs/base.yml",
            f"model_name={MODEL_NAME}",
            f"base_output_directory={MODEL_CHECKPOINT_PATH}",
            "use_multimodal=false",
            "scan_layers=true",
            "skip_jax_distributed_system=True",
        ],
        check=True,
        env=env
    )

    MODEL_CHECKPOINT_PATH = os.path.join(MODEL_CHECKPOINT_PATH, "0/items")
else:
    print(f"Model checkpoint exists at {MODEL_CHECKPOINT_PATH}")

## MaxText Configurations

In [ ]:
# Load configuration for SFT training
config_argv = [
    "",
    f"{MAXTEXT_PKG_DIR}/configs/post_train/sft.yml",
    f"load_parameters_path={MODEL_CHECKPOINT_PATH}",
    f"model_name={MODEL_NAME}",
    f"steps={os.environ.get('CI_STEPS', '100')}",
    "per_device_batch_size=1",
    "max_target_length=1024",
    "learning_rate=2.0e-5",
    "weight_dtype=bfloat16",
    "dtype=bfloat16",
    "hf_path=HuggingFaceH4/ultrachat_200k",
    f"base_output_directory={BASE_OUTPUT_DIRECTORY}",
    f"run_name={RUN_NAME}",
    "profiler=xplane",
]

config = pyconfig.initialize_pydantic(config_argv)

print("✓ SFT configuration loaded:")
print(f" Model: {config.model_name}")
print(f" Training Steps: {config.steps}")
print(f" Output Directory: {config.base_output_directory}")

## SFT Training

In [ ]:
import traceback

print("=" * 60)
print("🚀 Starting SFT Training...")
print("=" * 60)

try:
    trainer, mesh = train_sft.train(config)
    print("\n" + "=" * 60)
    print("✅ Training Completed Successfully!")
    print("=" * 60)
    print(f"📁 Checkpoints saved to: {config.checkpoint_dir}")
except Exception:
    print("\n" + "=" * 60)
    print("❌Training Failed!")
    print("=" * 60)
    traceback.print_exc()
    print("\nFor troubleshooting, refer to the Common Pitfalls & Debugging section:")
    print("https://maxtext.readthedocs.io/en/latest/guides/run_python_notebook.html#common-pitfalls-debugging")
    sys.exit(1)

## 📚 Learn More

- **CLI Usage**: https://maxtext.readthedocs.io/en/latest/tutorials/posttraining/sft.html
- **Configuration**: See `src/maxtext/configs/post_train/sft.yml` for all available options
- **Documentation**: Check `src/maxtext/trainers/post_train/sft/train_sft.py` for the `train` function implementation